# DELTA LAKE STORAGE

## 1. Delta Lake

#### What is Delta Lake?
* Delta Lake is an open-source storage layer that brings ACID transaction, scalable metadata handling, data versioning, time travel capabilities and unified support for streaming/batch processing to data lakes.
* It is default storage layer in Databricks.
* Delta Lake runs on top of existing Data Lakes (cloud storage like AWS S3, ADLF, etc.).
<br>
##### Features:
* ACID transaction
* Schema Enforcement
* Scalable Metadata handling
* DML Operations
* Data Versoning/ Time Travel
* Transaction Log
* Streaming/batch data processing

## 2. Understanding Traditional Data Storage Capabilities

### Data Warehouses

* Technical: Cleaned + Structured data
* Business: Operational data/ Analytical data
* Storage: Managed warehouse storage
* ACID transactions
* High Cost
* Schema-on-write
* Strong Data Modeling
* Analytical Processing/ Fast DB queries for Reporting
* Cons:
    * Less flexible
    * Expensive storage & compute
    * Data Preprocessing required
* Use Case: BI Reports/Dashboards


![image_1793511941367.png](./image_1793511941367.png "image_1793511941367.png")

### Data Lake
* Technical: Wide varity of data  
* Business: Central repository for all enterprise data
* Storage: Object Storage
* Flexible
* Low storage cost
* Schema-on-read
* Raw data is preserved (uncleaned)
* Usually built on object storage
* Very large scale  
* Use Case: Big Data, AI/ML, IoT, Analytics
* Cons:
    * Data Swamp
    * Lack of ACID Transactions

![image_1793511987586.png](./image_1793511987586.png "image_1793511987586.png")

## 3. Delta Table Architecture

* Delta Table is essentially Data files + Transactional Log + Metadata
* When a Delta Table is created, a new directory gets created in the specified storage location (usually in ADLS Gen2, S3, or local DBFS). This directory contains:
    * _delta_log (directory): Stores the metadata + transaction logs.
    * .parquet (files): The actual data stored in columnar format.
* With each commit a delta log file is created along with its corresponding parquet file. 

![image_1793520855468.png](./image_1793520855468.png "image_1793520855468.png")

### _delta_log

* all the transactional log files are stored in this _delta_log directory
* it tells Delta which Parquet files constitute the table at each version.
* these transacttional logs contains the metadata + transaction history
* this is what makes ACID + Time Travel possible
* this directory contains:
    * .json files: Every change to the table is recorded as a JSON file.
    * .checkpoint.parquet files: A compacted representation of the Delta transaction-log state.

#### .json log files



* _**Each json represents an action**_
* Delta json log file contains the actions and information associated with a particular table commit
* Json/metadata file contains info. related to actions and associated metadata for a particular commit, corresponsing parquet files, parititons and etc.
* Since the transaction log preserves the history of each commit, it makes Time Travel possible and helps optimize reads by allowing Delta to efficiently identify the relevant data files that need to be read.
* Whenever commit happens it creates a json file inside _delta_log directory that contains information related to that partuclar commit and corresponding parquet file. 
* _json log_ : _parquet_: 1:N  
* These actions describe what changed in that commit, such as:- 
    * add → a Parquet file was added to the logical table
    * remove → a Parquet file was removed from the logical table
    * metaData → table schema, partition columns, table properties, etc.
    * protocol → Delta reader/writer protocol requirements
    * commitInfo → information about the operation/commit
    * txn → transaction/application information when applicable

In [0]:
%sql
    SELECT commitInfo 
    FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
    WHERE commitInfo IS NOT NULL;

```
"commitInfo": { 
    "timestamp": 1587968586154,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "ErrorIfExists",
      "partitionBy": []
    },
    "isBlindAppend": true
  }
  ```  

In [0]:
%sql
SELECT metaData
FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
WHERE "metaData IS NOT NULL;


```
  "metaData": {
    "id": "5fba94ed-9794-4965-ba6e-6ee3c0d22af9",
    "format": {
      "provider": "parquet",
      "options": {}
    },
    "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}}]}",
    "partitionColumns": [],
    "configuration": {},
    "createdTime": 1587968585495
  ```

In [0]:
%sql
SELECT add
FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
WHERE add IS NOT NULL LIMIT 1;

```
{
      "path": "part-00000-a72b1fb3-f2df-41fe-a8f0-e65b746382dd-c000.snappy.parquet",
      "partitionValues": {},
      "size": 262,
      "modificationTime": 1587968586000,
      "dataChange": true
    }
```

#### .checkpoint.parquet

* **Snapshot of the Delta table's state at a particular version**
* After every n commits, a checkpoint .parquet is written to avoid reading thousands of JSON files.
* Over time, a Delta table can accumulate many JSON transaction-log files. Reading and replaying a large number of JSON files to reconstruct the current table state can become increasingly expensive. Therefore, Delta periodically creates a checkpoint, which stores the accumulated table state in a compact Parquet file. This allows Delta to start from the checkpoint and process only the JSON log files created after that checkpoint, reducing the number of files and I/O operations required to reconstruct the table state.
* A checkpoint action accumulates and compacts the actions from the Delta JSON logs, storing only the resulting state needed to represent a particular table version.

```text
JSON Logs
   ↓
ADD / REMOVE / UPDATE actions
   ↓
Accumulate + resolve actions
   ↓
Checkpoint
   ↓
Resulting state
   ↓
Table state at Version N
```

In [0]:
# Disable the safety check that forces format("delta")
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")
# Force read the individual parquet file
df_spark = spark.read.format("parquet").load("/Volumes/workspace/default/demo/00000000000000000002.checkpoint.parquet")
display(df_spark)
# Reset the configuration back to true for safety
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "true")


In [0]:
# Here we can see all the 'columns' that contains all the actions after being compacted
df_spark.columns
# ['txn', 'add', 'remove', 'metaData', 'protocol', 'domainMetadata']

In [0]:
df_spark.select('metaData').where(df_spark.metaData.isNotNull()).limit(1).toPandas()['metaData'].tolist()
#

Here is a snapshot from "add" column from the **checkpoint** file
```text
{'path': 'part-00000-1e14ba22-3114-46d1-96fb-48b4912507ce-c000.snappy.parquet',
  'partitionValues': {},
  'size': 5072,
  'modificationTime': 1723768498986,
  'dataChange': False,
  'tags': None,
  'deletionVector': None,
  'baseRowId': None,
  'defaultRowCommitVersion': None,
  'clusteringProvider': None,
  'stats': '{"numRecords":1,"minValues":{"id":1},"maxValues":{"id":1},"nullCount":{"id":0,"v":0,"array_of_variants":0,"struct_of_variants":{"v":0},"map_of_variants":0,"array_of_struct_of_variants":0,"struct_of_array_of_variants":{"v":0}}}'}
 ```

Here is a snapshot from "metaData" column from the **checkpoint** file
```
{'id': 'd7eb0848-b002-4e0b-9d8d-dd335c90946f',
  'name': None,
  'description': None,
  'format': {'provider': 'parquet', 'options': {}},
  'schemaString': '{"type":"struct","fields":[{"name":"id","type":"long","nullable":true,"metadata":{}},{"name":"v","type":"variant","nullable":true,"metadata":{}},{"name":"array_of_variants","type":{"type":"array","elementType":"variant","containsNull":true},"nullable":true,"metadata":{}},{"name":"struct_of_variants","type":{"type":"struct","fields":[{"name":"v","type":"variant","nullable":true,"metadata":{}}]},"nullable":true,"metadata":{}},{"name":"map_of_variants","type":{"type":"map","keyType":"string","valueType":"variant","valueContainsNull":true},"nullable":true,"metadata":{}},{"name":"array_of_struct_of_variants","type":{"type":"array","elementType":{"type":"struct","fields":[{"name":"v","type":"variant","nullable":true,"metadata":{}}]},"containsNull":true},"nullable":true,"metadata":{}},{"name":"struct_of_array_of_variants","type":{"type":"struct","fields":[{"name":"v","type":{"type":"array","elementType":"variant","containsNull":true},"nullable":true,"metadata":{}}]},"nullable":true,"metadata":{}}]}',
  'partitionColumns': array([], dtype=object),
  'configuration': {'delta.checkpointInterval': '2'},
  'createdTime': 1723768495302}
  ```

### .parquet Data file

* Here is where actual data is stored in parquet files
* **Delta never modified files - only adds new ones** i.e. Old files are NOT deleted on UPDATE/DELETE. New files are added; old ones are tombstoned in the log.
* Parquet is a column-oriented storage format designed for efficient querying through inherent optimization techniques like column pruning

In [0]:
# Disable the safety check that forces format("delta")
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")
# Force read the individual parquet file
df_spark = spark.read.format("parquet").load("/Volumes/workspace/default/demo/part-00000-a496f40c-e091-413a-85f9-b1b69d4b3b4e-c000.snappy.parquet")
display(df_spark)
# Reset the configuration back to true for safety
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "true")

**Output upon querying the data in the .parquet file.**
| date | county | state | fips | cases | deaths |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1/21/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/22/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/23/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/24/2020 | Cook | Illinois | 17031 | 1 | 0 |
| 1/24/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/25/2020 | Orange | California | 6059 | 1 | 0 |
| 1/25/2020 | Cook | Illinois | 17031 | 1 | 0 |
| 1/25/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/26/2020 | Maricopa | Arizona | 4013 | 1 | 0 |


### What happens in the background?

#### Creating a Delta Table:
```
CREATE TABLE employees (
    id INT,
    name STRING,
    department STRING,
    salary INT
) USING DELTA;
```
When the table is created:
* Databricks creates the Delta table structure/directory in storage - folder name same as table name.
* Directory will have only __delta_log_ directory.
* a JSON file named 00000000000000000000.json is written to __delta_log/_ that contains metadata/schema and properties about the table.
* Parquet data files = none


#### Inserting Data into a Delta Table:
```
INSERT INTO employees VALUES
(1, 'Alice', 'IT', 100000),
(2, 'Bob', 'HR', 80000),
(3, 'Charlie', 'IT', 90000);
```
When data is inserted into a Delta Table:
* Writes the data as parquet file/s  in the Delta Table directory: /mnt/delta/stock_data/part-00000-tid-12345.parquet
* Updates the transaction log with a new JSON entry: 00000000000000000001.json -> _delta_log/
* Supporse we execute another INSERT, Delta will create another parquet file with corresponding json log instead of modifying exisiting file.

#### Updating rows in a Delta Table
```
UPDATE employees
SET salary = 110000
WHERE id = 1;
```
When data is updated into a Delta Table:
* Identifies the existing Parquet file containing the record -> /mnt/delta/stock_data/part-00000-abc123.parquet/
* Reads the data, modifies the necessary rows, and writes a new Parquet file. -> /mnt/delta/stock_data/part-00002-new456.parquet/ & 00000000000000000002.json
* Marks the old file as removed in the transaction log.
```
{
  "remove": {
    "path": "part-00000-abc123.parquet"
  }
}
```
* Note: 
    * The old file is marked as removed but not physically deleted (this is handled during compaction).
    * It follows a copy-on-write (COW) mechanism.

#### Deleting rows in a Delta Table
```
DELETE FROM employees
WHERE id = 2;
```
When data is deleted into a Delta Table:
* Identifies the Parquet file containing the record.
* Writes a new Parquet file excluding the deleted row.
* Marks the old file as removed in the transaction log.

## 4. DELTA LAKE OPERATIONS

### 1. CREATE DELTA TABLE

#### Brief Description

* Defines the table schema and metadata.
* The table can be created as managed or external.
* An empty table generally has no business-data Parquet files yet.
* Can create a table from an existing Parquet dataset.
* Can also create a table using CTAS (CREATE TABLE AS SELECT).

#### Syntax

**SparkSQL**
```
CREATE TABLE <table_name> (
    column_name1 data_type(),
    column_name2 data_type(),
    ....
)
USING DELTA;
```

**PySpark (DataFrame API)**
```
#1. Define the schema for your empty table
schema = StructType([
    StructField("<colume_name1>", DataType(), True),
    StructField("<colume_name2>", DataType(), True),
    ...
])

#2. Create an empty DataFrame using the schema
empty_df = spark.emptyDataFrame(schema)

#3. Save the empty DataFrame as a Delta table
empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.my_empty_delta_table") 
```

#### Example

In [0]:
%sql
-- SparkSQL Syntax: To create delta table employees
CREATE TABLE employees (
    id INT,
    name STRING,
    department STRING,
    salary DECIMAL(10,2)
)
USING DELTA;

In [0]:
# PySpark Syntax: To create delta table employees
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Define the schema for your empty table
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("rate", IntegerType(), True),
    StructField("quantity", IntegerType(), True)
])

# 2. Create an empty DataFrame using the schema
orders_df = spark.emptyDataFrame(schema)

# 3. Save the empty DataFrame as a Delta table
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.orders")

In [0]:
from pyspark.sql import functions as F

orders_description = spark.sql("DESCRIBE EXTENDED orders").alias("orders")
employees_description = spark.sql("DESCRIBE EXTENDED employees").alias("employees")

result = (
    orders_description
    .join(
        employees_description,
        on="col_name",
        how="full_outer"
    )
    .select(
        F.col("col_name"),
        F.col("orders.data_type").alias("data_type_orders"),
        F.col("employees.data_type").alias("data_type_employees")
    )
)

display(result)

#### Code Variants for Specific Scenarios

##### Create using CTAS

In [0]:
%sql
CREATE TABLE employees_staging
USING DELTA
AS
SELECT *
FROM employees;

##### Create with partitioning

In [0]:
%sql
CREATE TABLE employees (
    id INT,
    name STRING,
    department STRING,
    salary DECIMAL(10,2),
    year INT
)
USING DELTA
PARTITIONED BY (year);

##### Create an external Delta table

In [0]:
%sql
CREATE TABLE employees(
    id INT,
    name STRING,
    department STRING,
    salary DECIMAL(10,2),
    year INT
)
USING DELTA
LOCATION '/mnt/data/employees';

##### Create table from Parquet

In [0]:
%sql
CREATE TABLE employees
USING DELTA
AS
SELECT *
FROM parquet.`/mnt/data/employees_parquet`;

### 2. INSERT INTO DELTA TABLE

#### Brief Description

* Existing rows are not modified.
* New Parquet data files are typically created.
* The transaction log records ADD actions for the new files.

#### Syntax

**SparkSQL**
```
INSERT INTO <table_name>
VALUES (...);
```


**PySpark (DataFrame API)**
```
new_rows_dataframe.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("<table_name>")
```

#### Example

In [0]:
%sql
INSERT INTO employees
VALUES
(1, 'Charlie', 'Finance', 90000),
(2, 'David', 'IT', 95000),
(3, 'Philip', 'Migration', 80000);

SELECT * FROM employees

In [0]:
# PySpark Syntax: To create delta table employees
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Define the schema for your empty table
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("rate", IntegerType(), True),
    StructField("quantity", IntegerType(), True)
])

df = spark.createDataFrame([(1, 100, 10), (1, 150, 5), (2, 53, 23), (3, 190, 12)],schema)

df.write.format('delta')\
    .mode('append')\
        .saveAsTable('Orders')

spark.read.format('delta').table('Orders').show()

#### Variants

##### Insert from another table

In [0]:
%sql
INSERT INTO employees
SELECT id, name, department, salary
FROM employees_staging;

##### Insert specific columns

In [0]:
%sql
INSERT INTO employees (id, name, department, salary)
VALUES (6, 'Frank', 'IT', 92000);

##### Insert using a transformation

In [0]:
%sql
INSERT INTO employees
SELECT
    id,
    UPPER(name),
    department,
    salary
FROM employees_staging;

### 3. UPDATE DELTA TABLE

#### Brief Descriptions

- Modifies existing rows.
- Uses a SET clause to specify new values.
- Uses WHERE to determine which rows are affected.
- Affected data may be rewritten into new Parquet files.
- Transaction log records the resulting file changes, typically through REMOVE + ADD.
- Supports updating one or multiple columns.
- Can update values using expressions or values derived from another table.

#### Syntax

**PySpark**
```
UPDATE <table_name>
SET <column_name> = <value>
WHERE <condition>;
```
**PySpark (DataFrame API)**
```
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "<table_name>"
)

delta_table.update(
    condition="<condition>",
    set={
        "<column_name>": "<value>"
    }
)
```

#### Example

In [0]:
%sql
UPDATE employees
SET salary = 110000
WHERE id = 1;

SELECT * FROM employees WHERE id = 1;

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "orders")

delta_table.update(
    condition="id = 2",
    set={"rate": "55"}
)

display(spark.read.format('delta').table('orders'))

#### Variants

##### Update multiple columns

In [0]:
%sql
UPDATE employees
SET
    department = 'Engineering',
    salary = 120000
WHERE id = 1;

##### Update all rows

In [0]:
%sql
UPDATE employees
SET salary = salary * 1.10;

##### Update using another column

In [0]:
%sql
UPDATE employees
SET salary = salary + 5000
WHERE department = 'IT';

##### Update using a subquery

In [0]:
%sql
UPDATE employees
SET department = (
    SELECT department
    FROM employee_updates
    WHERE employee_updates.id = employees.id
)
WHERE id IN (
    SELECT id
    FROM employee_updates
);

### 4. DELETE FROM DELTA TABLE

####Brief Description

- Removes rows from the logical Delta table.
- Delta generally does not physically delete individual rows from an existing Parquet file.
- Affected files can be rewritten.
- Transaction log records the corresponding file-state changes.
- Old physical files can remain until eligible cleanup with VACUUM.
- Supports conditional and full-table deletion.

####Syntax


**SparkSQL**
```
DELETE FROM <table_name>
WHERE <condition>;
```

**PySpark (DataFrame API)**
```
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "<table_name>"
)

delta_table.delete(
    condition="<condition>"
)
```

#### Example

In [0]:
%sql
DELETE FROM employees
WHERE id = 3;

SELECT * FROM employees

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, 'orders')

delta_table.delete('id = 3')

spark.read.table('orders').display()

#### Variants

##### Delete multiple rows

In [0]:
DELETE FROM employees
WHERE department = 'HR';

##### Delete based on multiple conditions

In [0]:
DELETE FROM employees
WHERE department = 'IT'
AND salary < 80000;

##### Delete all rows

In [0]:
DELETE FROM employees;

##### Delete using a subquery

In [0]:
DELETE FROM employees
WHERE id IN (
    SELECT id
    FROM employees_to_remove
);

### 5. MERGE INTO DELTA TABLE

#### Brief Description

- Used for upsert scenarios.
- Can perform INSERT, UPDATE, and DELETE in a single operation.
- Compares a source dataset with a target Delta table.
- Uses a matching condition.
- WHEN MATCHED handles existing records.
- WHEN NOT MATCHED handles new records.
- WHEN NOT MATCHED BY SOURCE can handle target records without a corresponding source record.

#### Syntax

**SparkSQL**
```
MERGE INTO <target_table_name> AS target
USING <source_table_name> AS source
ON target.id = source.id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;
```
**PySpark (DataFrame API)**
```
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "<target_table_name>"
)

source = spark.table("<source_table_name>")

(
    target.alias("t")
    .merge(
        source.alias("s"),
        "t.id = s.id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
```

#### Example

In [0]:
%sql
CREATE TABLE employees_source
(id INT,
name STRING,
department STRING,
salary INT);

INSERT INTO employees_source 
VALUES
(3, 'John', 'Sales', 50000),
(2, 'Jane', 'Marketing', 60000),
(4, 'Bob', 'Engineering', 70000);

MERGE INTO employees AS target 
USING employees_source AS source
ON target.id = source.id 

WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN 
INSERT *;

SELECT * FROM employees;

In [0]:
%sql
SELECT * FROM orders;

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable

schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('rate', IntegerType(), True),
    StructField('quantity', IntegerType(), True)])

data = [
    (3, 28, 27),
    (2, 25, 78)]

df = spark.createDataFrame(schema=schema,data=data)

#df.write.format('delta').mode('append').saveAsTable('orders_source')

delta_table = DeltaTable.forName(spark, 'orders')

delta_table.alias('target').merge(
    df.alias('source'),
    ' target.id = source.id'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


display(spark.read.format('delta').table('orders'))

#### Variant

##### Update existing + insert new

In [0]:
MERGE INTO employees AS t
USING employee_updates AS s
ON t.id = s.id

WHEN MATCHED THEN
    UPDATE SET
        t.name = s.name,
        t.department = s.department,
        t.salary = s.salary

WHEN NOT MATCHED THEN
    INSERT (
        id,
        name,
        department,
        salary
    )
    VALUES (
        s.id,
        s.name,
        s.department,
        s.salary
    );

##### Conditional update

In [0]:
MERGE INTO employees AS t
USING employee_updates AS s
ON t.id = s.id

WHEN MATCHED AND s.salary > t.salary THEN
    UPDATE SET
        t.salary = s.salary

WHEN NOT MATCHED THEN
    INSERT *;

##### Update + insert + delete

In [0]:
MERGE INTO employees AS t
USING employee_updates AS s
ON t.id = s.id

WHEN MATCHED AND s.operation = 'DELETE' THEN
    DELETE

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;

### 6. CREATE OR REPLACE TABLE

#### Brief Description:

- Replaces the definition and data of a table with a new table definition/query result.
- Particularly useful for full-refresh pipelines.
- With Delta, replacement is recorded transactionally.
- Should not be confused with DROP TABLE followed by CREATE TABLE. (Internally object pointer remains and the swap happens instantly.)

#### Example

In [0]:
df = spark.table("employees_staging")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("employees_dummy")

In [0]:
%sql
CREATE OR REPLACE TABLE employees_dummy
USING DELTA
AS
SELECT *
FROM employees_staging;

### 7. TRUNCATE TABLE

#### Brief Definition

- Removes all rows from a Delta table.
- Useful when you need to empty a table before reloading it.
- It is different from DELETE FROM table conceptually and operationally.

#### Example

In [0]:
%sql

TRUNCATE TABLE employees_dummy;
SELECT * FROM employees_dummy;

### 8. DROP TABLE

#### Brief Definition

- Removes the table definition/metadata from the catalog.
- For managed tables, associated data is generally removed according to the platform's managed-table behavior.
- For external tables, dropping the table generally removes the catalog registration, not the underlying data files.

#### Example

In [0]:
%sql
DROP TABLE IF EXISTS employees_dummy;

### 9. ALTER TABLE

### Brief Description

- Changes the structure or metadata of an existing Delta table.
- Can change table properties.
- Can rename a table.
- Can modify partition-related/table metadata in supported scenarios.
- Changes are recorded in Delta table metadata.

#### Syntax

**SparkSQL**
```
ALTER TABLE <table_name>
RENAME TO <new_table_name> -- Renaming table name
ADD COLUMNS (<column_nanme> <data_type>,..) -- Adding columns
SET TBLPROPERTIES ('<property_name>'= '<value>'); -- Setting table properties
UNSET TBLPROPERTIES ('<property_name>'); -- Removing a properties
;
```

### Example

In [0]:
%sql
ALTER TABLE employees_dummy
ADD COLUMNS (
    email STRING
);

SELECT * FROM employees_dummy;

#### Variants

##### Add column

In [0]:
%sql
ALTER TABLE employees
ADD COLUMNS (
    email STRING
);

##### Add multiple columns

In [0]:
%sql
ALTER TABLE employees
ADD COLUMNS (
    email STRING,
    phone STRING
);

#####Rename table

In [0]:
%sql
ALTER TABLE employees
RENAME TO employees_new;

#####Set table property

In [0]:
%sql
ALTER TABLE employees
SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);

#####Unset table property

In [0]:
%sql
ALTER TABLE employees
UNSET TBLPROPERTIES (
    'delta.enableChangeDataFeed'
);

###10. DESCRIBE TABLE

DESCRIBE:
- Displays table metadata.
- Useful for understanding the schema and table properties.
- Can show columns, data types, comments, and other metadata depending on the command.

DESCRIBE HISTORY:
- Displays the transaction history of a Delta table.
- Shows operations such as WRITE, UPDATE, DELETE, MERGE, OPTIMIZE, etc.
- Shows table versions.
- Each successful transaction creates a new Delta table version.

#### Syntax

**SparkSQL**
```
DESCRIBE TABLE <table_name>; --  Returns the table's column schemas and data types;
DESCRIBE DETAIL <table_name>; -- Provides structural metadata like size, file count, and partition columns
DESCRIBE HISTORY <table_name>; --  outputs the chronological audit log of writes, updates, and operations for time travel
```

#### Example

In [0]:
%sql
DESCRIBE TABLE employees;

In [0]:
%sql
DESCRIBE DETAIL employees;

In [0]:
%sql
DESCRIBE HISTORY employees;

### 11. OPTIMIZE

#### Brief Description

- Compacts small Parquet files into larger files
- Improves read performance by reducing the number of files that must be opened/read
- Creates new optimized files
- Old files become logically obsolet
- Databricks-specific Delta operation
- Databricks can automatically apply optimization features to Delta tables, depending on the feature and table type:
    - Predictive Optimization
    - Auto Compaction (autoOptimize.autoCompact)
    - Optimized Writes (autoOptimize.optimizeWrite)
- Z-Ordering is used to colocate related data in the same set of files
- Z-ORDER clause extends this by physically reorganizing rows based on the specified column values so that similar data points live in the same storage blocks
- Z-Ordering is useful for frequently filtered columns
- It works with Delta's data-skipping statistics

**Bin Packaging**
![image_1793738756838.png](./image_1793738756838.png "image_1793738756838.png")

**Z-Ordering**
![image_1793742106229.png](./image_1793742106229.png "image_1793742106229.png")

#### Syntax

**SparkSQL**
```
--Bin Packaging
OPTIMIZE <table_name>;

--Z-Order
OPTIMIZE <table_name>
ZORDER BY (<column_names>, <column_names>);
```

#### Example

In [0]:
%sql
OPTIMIZE orders;

In [0]:
%sql
DESCRIBE HISTORY employees;
-- As you can see OPTIMIZE has already been applied earlier automatically. This is because of Predictive Optimization.

|version|timestamp|userId|userName|operation|operationParameters|job|notebook|queryHistoryStatementId|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics|userMetadata|engineInfo|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|15|2026-11-02T14:55:20.000Z|78621438202630|sandipan.kar.data@gmail.com|OPTIMIZE|{"clusterBy":"[]","zOrderBy":"[]","batchId":"0","predicate":"[]","auto":"true"}|null|{"notebookId":"1227692749603692"}|aa8695df-32bc-4965-a1cc-28f0b3d6f750|1102-140438-qnztfp7j-v2n|14|SnapshotIsolation|false|{"numRemovedFiles":"3","numRemovedBytes":"3782","p25FileSize":"1294","numDeletionVectorsRemoved":"1","minFileSize":"1294","p75FileSize":"1294","p50FileSize":"1294","numAddedBytes":"1294","numAddedFiles":"1","maxFileSize":"1294"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|14|2026-11-02T14:55:19.000Z|78621438202630|sandipan.kar.data@gmail.com|MERGE|{"predicate":"[\"(id#13634 = id#13646)\"]","clusterBy":"[]","matchedPredicates":"[{\"actionType\":\"update\"}]","statsOnLoad":"false","notMatchedBySourcePredicates":"[]","notMatchedPredicates":"[{\"actionType\":\"insert\"}]"}|null|{"notebookId":"1227692749603692"}|aa8695df-32bc-4965-a1cc-28f0b3d6f750|1102-140438-qnztfp7j-v2n|13|WriteSerializable|false|{"numTargetRowsCopied":"0","numTargetRowsDeleted":"0","numTargetBytesRemoved":"0","numTargetDeletionVectorsAdded":"1","numTargetRowsMatchedUpdated":"3","numTargetRowsMatchedDeleted":"0","numTargetRowsUpdated":"3","numTargetChangeFilesAdded":"0","numTargetRowsNotMatchedBySourceDeleted":"0","rewriteTimeMs":"1698","numTargetFilesAdded":"2","numTargetBytesAdded":"2488","executionTimeMs":"2794","materializeSourceTimeMs":"102","numTargetRowsInserted":"0","numTargetDeletionVectorsUpdated":"0","scanTimeMs":"955","numOutputRows":"3","numTargetDeletionVectorsRemoved":"0","numTargetRowsNotMatchedBySourceUpdated":"0","numSourceRows":"2","numTargetFilesRemoved":"0"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|13|2026-11-02T14:54:10.000Z|78621438202630|sandipan.kar.data@gmail.com|OPTIMIZE|{"clusterBy":"[]","zOrderBy":"[]","batchId":"0","predicate":"[]","auto":"true"}|null|{"notebookId":"1227692749603692"}|771a5e76-ede6-460e-936d-9a97098ec29d|1102-140438-qnztfp7j-v2n|12|SnapshotIsolation|false|{"numRemovedFiles":"5","numRemovedBytes":"6237","p25FileSize":"1294","numDeletionVectorsRemoved":"1","minFileSize":"1294","p75FileSize":"1294","p50FileSize":"1294","numAddedBytes":"1294","numAddedFiles":"1","maxFileSize":"1294"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|12|2026-11-02T14:54:08.000Z|78621438202630|sandipan.kar.data@gmail.com|MERGE|{"predicate":"[\"(id#12992 = id#13004)\"]","clusterBy":"[]","matchedPredicates":"[{\"actionType\":\"update\"}]","statsOnLoad":"true","notMatchedBySourcePredicates":"[]","notMatchedPredicates":"[{\"actionType\":\"insert\"}]"}|null|{"notebookId":"1227692749603692"}|771a5e76-ede6-460e-936d-9a97098ec29d|1102-140438-qnztfp7j-v2n|11|WriteSerializable|false|{"numTargetRowsCopied":"0","numTargetRowsDeleted":"0","numTargetBytesRemoved":"0","numTargetDeletionVectorsAdded":"1","numTargetRowsMatchedUpdated":"2","numTargetRowsMatchedDeleted":"0","numTargetRowsUpdated":"2","numTargetChangeFilesAdded":"0","numTargetRowsNotMatchedBySourceDeleted":"0","rewriteTimeMs":"2254","numTargetFilesAdded":"4","numTargetBytesAdded":"4955","executionTimeMs":"3718","materializeSourceTimeMs":"301","numTargetRowsInserted":"3","numTargetDeletionVectorsUpdated":"0","scanTimeMs":"1114","numOutputRows":"5","numTargetDeletionVectorsRemoved":"0","numTargetRowsNotMatchedBySourceUpdated":"0","numSourceRows":"4","numTargetFilesRemoved":"0"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|11|2026-11-02T14:07:56.000Z|78621438202630|sandipan.kar.data@gmail.com|OPTIMIZE|{"clusterBy":"[]","zOrderBy":"[]","batchId":"0","predicate":"[]","auto":"true"}|null|{"notebookId":"1227692749603692"}|564119bf-278f-4a86-b465-b1920952f551|1102-140438-qnztfp7j-v2n|10|SnapshotIsolation|false|{"numRemovedFiles":"1","numRemovedBytes":"1286","p25FileSize":"1282","numDeletionVectorsRemoved":"1","minFileSize":"1282","p75FileSize":"1282","p50FileSize":"1282","numAddedBytes":"1282","numAddedFiles":"1","maxFileSize":"1282"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|10|2026-11-02T14:07:54.000Z|78621438202630|sandipan.kar.data@gmail.com|DELETE|{"predicate":"[\"(id#10881 = 3)\"]"}|null|{"notebookId":"1227692749603692"}|564119bf-278f-4a86-b465-b1920952f551|1102-140438-qnztfp7j-v2n|9|WriteSerializable|false|{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","executionTimeMs":"1278","numDeletionVectorsUpdated":"0","numAddedFiles":"0","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"0","numDeletedRows":"1","scanTimeMs":"773","numAddedBytes":"0","rewriteTimeMs":"505"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|9|2026-11-02T12:35:19.000Z|78621438202630|sandipan.kar.data@gmail.com|OPTIMIZE|{"clusterBy":"[]","zOrderBy":"[]","batchId":"0","predicate":"[]","auto":"true"}|null|{"notebookId":"1227692749603692"}|522b8e82-179f-4c02-ac32-2b338c53b32e|1102-122426-blxdsv3h-v2n|8|SnapshotIsolation|false|{"numRemovedFiles":"2","numRemovedBytes":"2524","p25FileSize":"1286","numDeletionVectorsRemoved":"1","minFileSize":"1286","p75FileSize":"1286","p50FileSize":"1286","numAddedBytes":"1286","numAddedFiles":"1","maxFileSize":"1286"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|8|2026-11-02T12:35:17.000Z|78621438202630|sandipan.kar.data@gmail.com|UPDATE|{"predicate":"[\"(id#10861 = 2)\"]"}|null|{"notebookId":"1227692749603692"}|522b8e82-179f-4c02-ac32-2b338c53b32e|1102-122426-blxdsv3h-v2n|7|WriteSerializable|false|{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","executionTimeMs":"2090","numDeletionVectorsUpdated":"0","scanTimeMs":"723","numAddedFiles":"1","numUpdatedRows":"1","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"0","numAddedBytes":"1234","rewriteTimeMs":"1358"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|7|2026-11-01T19:33:10.000Z|78621438202630|sandipan.kar.data@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|{"notebookId":"1227692749603692"}|67de2de3-bda7-4863-a50c-bf622a6ca8e6|1101-190930-zi4o0cu6-v2n|6|WriteSerializable|true|{"numFiles":"1","numOutputRows":"4","numOutputBytes":"1290"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|6|2026-11-01T19:33:01.000Z|78621438202630|sandipan.kar.data@gmail.com|TRUNCATE|{}|null|{"notebookId":"1227692749603692"}|b4c60952-31c9-47e9-98b7-61dec4f02866|1101-190930-zi4o0cu6-v2n|5|WriteSerializable|false|{"numRemovedFiles":"3","numRemovedBytes":"3870","numDeletionVectorsRemoved":"0","executionTimeMs":"59","numDeletedRows":"12"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|5|2026-11-01T19:32:35.000Z|78621438202630|sandipan.kar.data@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|{"notebookId":"1227692749603692"}|ae2ea961-b026-4d80-84c8-e67858de259e|1101-190930-zi4o0cu6-v2n|4|WriteSerializable|true|{"numFiles":"1","numOutputRows":"4","numOutputBytes":"1290"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|4|2026-11-01T19:32:30.000Z|78621438202630|sandipan.kar.data@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|{"notebookId":"1227692749603692"}|2872a26e-c642-4fac-a1d6-eca673a5ea7d|1101-190930-zi4o0cu6-v2n|3|WriteSerializable|true|{"numFiles":"1","numOutputRows":"4","numOutputBytes":"1290"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|3|2026-11-01T19:30:47.000Z|78621438202630|sandipan.kar.data@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|{"notebookId":"1227692749603692"}|49b03ec6-4087-40b9-9429-3231c2f7624a|1101-190930-zi4o0cu6-v2n|2|WriteSerializable|true|{"numFiles":"1","numOutputRows":"4","numOutputBytes":"1290"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|2|2026-11-01T19:10:16.000Z|78621438202630|sandipan.kar.data@gmail.com|CREATE OR REPLACE TABLE AS SELECT|{"isV1SaveAsTableOverwrite":"true","partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"true","properties":"{\"delta.parquet.format.version\":\"2.12.0\",\"delta.parquet.format.version.afe.internal\":\"2.12.0\",\"delta.parquet.compression.codec\":\"zstd\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"true"}|null|{"notebookId":"1227692749603692"}|54b3f094-af8f-4102-8d80-56d5a8f2f33f|1101-190930-zi4o0cu6-v2n|1|WriteSerializable|false|{"numFiles":"0","numRemovedFiles":"0","numRemovedBytes":"0","numDeletionVectorsRemoved":"0","numOutputRows":"0","numOutputBytes":"0"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|1|2026-11-01T19:10:02.000Z|78621438202630|sandipan.kar.data@gmail.com|CREATE OR REPLACE TABLE AS SELECT|{"isV1SaveAsTableOverwrite":"true","partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"true","properties":"{\"delta.parquet.format.version\":\"2.12.0\",\"delta.parquet.format.version.afe.internal\":\"2.12.0\",\"delta.parquet.compression.codec\":\"zstd\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"true"}|null|{"notebookId":"1227692749603692"}|7b66b936-f8e8-407a-b1a6-de9da061a7bf|1101-190930-zi4o0cu6-v2n|0|WriteSerializable|false|{"numFiles":"0","numRemovedFiles":"0","numRemovedBytes":"0","numDeletionVectorsRemoved":"0","numOutputRows":"0","numOutputBytes":"0"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|
|0|2026-11-01T15:38:43.000Z|78621438202630|sandipan.kar.data@gmail.com|CREATE OR REPLACE TABLE AS SELECT|{"isV1SaveAsTableOverwrite":"true","partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"true","properties":"{\"delta.parquet.format.version\":\"2.12.0\",\"delta.parquet.format.version.afe.internal\":\"2.12.0\",\"delta.parquet.compression.codec\":\"zstd\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"true"}|null|{"notebookId":"1227692749603692"}|1969c8e0-be04-4103-a2c5-df031a57a907|1101-143438-8twwsz7f-v2n|null|WriteSerializable|false|{"numFiles":"0","numRemovedFiles":"0","numRemovedBytes":"0","numDeletionVectorsRemoved":"0","numOutputRows":"0","numOutputBytes":"0"}|null|Databricks-Runtime/19.12.x-aarch64-photon-scala2.13|

In [0]:
%sql
DESCRIBE TABLE EXTENDED orders;
-- SHOW TBLPROPERTIES orders ('delta.autoOptimize.autoCompact');
-- Here in the DESCRIBE TABLE EXTENDED output Predictive Optimization is ENABLED

|col_name|data_type|comment|
|---|---|---|
|id|int|null|
|rate|int|null|
|quantity|int|null|
||||
|# Delta Statistics Columns|||
|Column Names|id, rate, quantity||
|Column Selection Method|first-32||
||||
|# Detailed Table Information|||
|Catalog|workspace||
|Database|default||
|Table|orders||
|Created Time|Sun Nov 01 15:38:43 UTC 2026||
|Last Access|UNKNOWN||
|Created By|Spark ||
|Statistics|6245 bytes, 7 rows||
|Type|MANAGED||
|Location|||
|Provider|delta||
|Owner|sandipan.kar.data@gmail.com||
|Is_managed_location|true||
|Predictive Optimization|ENABLE (inherited from METASTORE metastore_aws_us_east_2)||
|Table Properties|[delta.enableDeletionVectors=true,delta.feature.appendOnly=supported,delta.feature.deletionVectors=supported,delta.feature.invariants=supported,delta.minReaderVersion=3,delta.minWriterVersion=7,delta.parquet.compression.codec=zstd,delta.parquet.format.version=2.12.0,delta.parquet.format.version.afe.internal=2.12.0]||

#### Variants

##### Optimize a specific partition

In [0]:
%sql
OPTIMIZE orders
WHERE year = 2026;

##### Optimize with Z-ORDER

In [0]:
%sql
OPTIMIZE employees
ZORDER BY (id, department);

##### Optimize a filtered subset with Z-ORDER

In [0]:
%sql
OPTIMIZE sales
WHERE year = 2026
ZORDER BY (customer_id);

### 12. VACUUM

#### Brief Description

- Physically removes obsolete Parquet files that are older than the configured retention threshold.
- Does not determine the logical table state.
- VACUUM cleans up physical files that are no longer required.
- Can affect the ability to time travel to old versions.
- RETAIN specifies the exact cutoff window for historical data files that must be kept.
- Retention settings should be handled carefully in production.
- This is a physical cleanup operation, not a normal row-level DELETE.
- DRY RUN allows you to inspect files that would be removed without actually deleting them.

![image_1793765829083.png](./image_1793765829083.png "image_1793765829083.png")

#### Syntax

**PySpark**
```
VACUUM <table_name> [RETAIN <hours> HOURS];
```

#### Example

In [0]:
%sql 
VACUUM employees DRY RUN;

In [0]:
%sql
VACUUM employees RETAIN 168 HOURS;

### 13. TIME TRAVEL

### 14. RESTORE DELTA TABLE

### 15. CLONE DELTA TABLE

### 16. READ DELTA TABLE

### 17. SCHEMA EVOLUTION

Important Links:
* https://github.com/delta-io/delta-rs/tree/main/crates/test/tests/data/simple_table
* https://medium.com/@jithujosekokken/understanding-delta-tables-in-databricks-a-deep-dive-into-the-transaction-log-77c31467b99f
* https://pyspark.in/tutorials/spark/Delta%20Lake/868
* https://docs.databricks.com/aws/en/delta/tutorial?language=Python